# V2: Advanced Pitcher Strikeout Prediction Model
In this version, we will significantly upgrade our model by adding more parameters. Specifically:
1. **Filtering out Spring Training Games**
2. **Ensuring only Pitcher Starts**
3. **Home vs. Away Advantage**
4. **Exact Lineup Handedness (L/R/S)** for the opposing team
5. **Pitcher Handedness** (Left or Right-handed pitcher)

### Step 1: Connect & Extract Pitching Data
We query the `event_boxscores_pitching` table, joined with `events` (to exclude Spring Training) and `event_competitors` (to determine Home/Away).

In [1]:
import pandas as pd
import sqlalchemy

engine = sqlalchemy.create_engine("postgresql:///mlb_db")

print("Pulling Pitching Data...")
query_pitching = """
SELECT 
    p.event_id,
    e.date,
    p.athlete_id as pitcher_id,
    p.team_id as pitcher_team_id,
    a.throws as pitcher_throws,
    c.home_away as pitcher_home_away,
    p.k as target_k
FROM event_boxscores_pitching p
JOIN events e ON p.event_id = e.event_id
JOIN athletes a ON p.athlete_id = a.athlete_id
JOIN event_competitors c ON p.event_id = c.event_id AND p.team_id = c.team_id
WHERE p.starter = true 
  AND e.type_id != 1  -- Exclude Spring Training
ORDER BY e.date ASC
"""
df_pitching = pd.read_sql(query_pitching, engine)
print(f"Loaded {len(df_pitching)} regular/postseason pitcher starts.")
df_pitching.head()

Pulling Pitching Data...
Loaded 245 regular/postseason pitcher starts.


,event_id,date,pitcher_id,pitcher_team_id,pitcher_throws,pitcher_home_away,target_k
0,401814702,2026-03-26 00:05:00,32685,10,L,away,4
1,401814702,2026-03-26 00:05:00,41216,26,R,home,7
2,401814693,2026-03-26 17:15:00,39825,21,R,home,7
3,401814693,2026-03-26 17:15:00,4719507,23,R,away,1
4,401814696,2026-03-26 18:10:00,5080761,8,R,home,11


### Step 2: Opponent Data & Lineup Handedness
We will query the `event_boxscores_batting` table for the *starting batters* only, and join with the `athletes` table to find out how many Lefties, Righties, and Switch Hitters the pitcher faced.

In [2]:
print("Pulling Starting Lineup Batting Data...")
query_batting = """
SELECT 
    b.event_id,
    e.date,
    b.team_id,
    SUM(b.k) as team_strikeouts,
    SUM(CASE WHEN a.bats = 'L' THEN 1 ELSE 0 END) as opp_start_lhb,
    SUM(CASE WHEN a.bats = 'R' THEN 1 ELSE 0 END) as opp_start_rhb,
    SUM(CASE WHEN a.bats = 'S' THEN 1 ELSE 0 END) as opp_start_shb
FROM event_boxscores_batting b
JOIN events e ON b.event_id = e.event_id
JOIN athletes a ON b.athlete_id = a.athlete_id
WHERE b.starter = true
GROUP BY b.event_id, e.date, b.team_id
ORDER BY e.date ASC
"""
df_batting = pd.read_sql(query_batting, engine)
df_batting.head()

Pulling Starting Lineup Batting Data...


,event_id,date,team_id,team_strikeouts,opp_start_lhb,opp_start_rhb,opp_start_shb
0,401078885,2019-02-22 18:05:00,22,5.0,2,4,0
1,401078885,2019-02-22 18:05:00,30,7.0,3,6,0
2,401078698,2019-02-22 20:10:00,11,5.0,3,5,0
3,401078698,2019-02-22 20:10:00,12,4.0,6,3,0
4,401078890,2019-02-23 18:05:00,1,4.0,3,4,0


### Step 3: Matchups (Who played who?)
Same as before, we connect the pitcher's team to the opposing team for that specific game.

In [3]:
query_matchups = """
SELECT event_id, team_id, home_away
FROM event_competitors
"""
df_matchups = pd.read_sql(query_matchups, engine)
game_teams = df_matchups.groupby('event_id')['team_id'].apply(list).to_dict()

def get_opp_team(row):
    teams = game_teams.get(row['event_id'], [])
    for t in teams:
        if t != row['pitcher_team_id']:
            return t
    return None

df_pitching['opp_team_id'] = df_pitching.apply(get_opp_team, axis=1)


### Step 4: Feature Engineering
We calculate the rolling historical averages *and* attach the lineup statistics for *today's* game (since the pitcher knows the lineup before throwing the first pitch!).

In [4]:
df_pitching = df_pitching.sort_values('date')
df_batting = df_batting.sort_values('date')

# 1. Pitcher's recent strikeout form (Last 5 starts)
df_pitching['pitcher_k_last_5'] = (
    df_pitching.groupby('pitcher_id')['target_k']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# 2. Opposing team's recent strikeout form (Last 10 games)
df_batting['team_k_last_10'] = (
    df_batting.groupby('team_id')['team_strikeouts']
    .transform(lambda x: x.shift(1).rolling(window=10, min_periods=1).mean())
)

# 3. Merge
df_model = pd.merge(
    df_pitching, 
    df_batting[['event_id', 'team_id', 'team_k_last_10', 'opp_start_lhb', 'opp_start_rhb', 'opp_start_shb']], 
    left_on=['event_id', 'opp_team_id'], 
    right_on=['event_id', 'team_id'], 
    how='inner'
)

# 4. Encode Categorical Data (Machine Learning models need numbers!)
# Convert 'home' to 1 and 'away' to 0
df_model['is_home'] = (df_model['pitcher_home_away'] == 'home').astype(int)

# Convert 'R' (Right) to 1 and 'L' (Left) to 0
df_model['is_rhp'] = (df_model['pitcher_throws'] == 'R').astype(int)

# Drop early season rows missing history
df_model = df_model.dropna(subset=['pitcher_k_last_5', 'team_k_last_10', 'target_k'])
print(f"Final Model Dataset Ready! Total Valid Matchups: {len(df_model)}")


Final Model Dataset Ready! Total Valid Matchups: 89


### Step 5: Training The Model with New Features

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# New massive feature list!
features = [
    'pitcher_k_last_5', 
    'team_k_last_10', 
    'is_home',          # Home-field advantage
    'is_rhp',           # Pitcher Handedness
    'opp_start_lhb',    # Number of Lefty Batters
    'opp_start_rhb',    # Number of Righty Batters
    'opp_start_shb'     # Number of Switch Hitters
]

X = df_model[features]
y = df_model['target_k']

# Chronological Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Train
model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

print("Model Trained Successfully!")

Model Trained Successfully!


### Step 6: Evaluation
Let's see if the extra context helped our Random Forest understand the game better.

In [6]:
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f} Strikeouts\n")

print("Feature Importances (Notice how it uses the new stats!):")
importances = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

for _, row in importances.iterrows():
    print(f"- {row['Feature']}: {row['Importance']:.2%}")


Mean Absolute Error (MAE): 2.76 Strikeouts

Feature Importances (Notice how it uses the new stats!):
- team_k_last_10: 30.84%
- pitcher_k_last_5: 30.35%
- opp_start_rhb: 19.56%
- opp_start_lhb: 15.59%
- is_home: 2.06%
- is_rhp: 1.59%
- opp_start_shb: 0.00%
